In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import pickle

import senepy as sp
import random

from collections import defaultdict
from scipy.optimize import brentq
from scipy.stats import norm
from scipy.stats import spearmanr
from sklearn.mixture import GaussianMixture
from matplotlib.lines import Line2D

In [ ]:
pd.set_option("display.max_rows", 100)

In [ ]:
for _, row in sp_hubs_.metadata.iterrows():
    key = tuple(row.iloc[:3])
    print(key, "\n", sp_hubs_.get_genes(key))

In [ ]:
adata = sc.read_h5ad("./data/adata_all_harmonized.h5ad")
adata.raw = adata.copy()

In [ ]:
with open("./data/senepy_denovo_signatures_code/ready/human_hubs.pkl", "rb") as file:
    calc_hubs = pickle.load(file)

with open(
    "./data/senepy_denovo_signatures_code/ready/human_signatures.pkl", "rb"
) as file:
    calc_sigs = pickle.load(file)

#### Calc sen scores

via sp.score_all_cells()

In [ ]:
score_colname = "sp_score_sigs"
adata.obs[score_colname] = np.nan

for (t, c), hub in tqdm(calc_sigs.items()):
    ct_mask = (adata.obs.Tissue_global == t) & (adata.obs.cell_type_final == c)
    scores = sp.score_all_cells(
        adata[ct_mask], hub=hub, identifiers=["GSE_id"], binarize=True
    )
    adata.obs.loc[ct_mask, score_colname] = scores

#### Plot score distr / GLM

In [ ]:
def plot_score_by_donor(
    obs: pd.DataFrame,
    cluster_col: str,
    cluster_val: list,
    score_col: str,
    min_cells: int = 100,
    threshold_method: str = "mean_nsd",  # 'mean_nsd'/'percentile'/me + iqr
    n_sd: float = 3,
    percentile: float = 75,
    draw_global: bool = True,
    global_threshold_kwargs: dict = None,
):
    """
    Стрипплот и статистики score_col по донорам для выбранного cluster_val в cluster_col.

    Параметры:
    - obs: DataFrame с колонками [GSM_id, GSE_id, Age, cluster_col, score_col, ...].
    - cluster_col: имя колонки с кластером/типом клетки.
    - cluster_val: нужное значение(я) в cluster_col в виде списка.
    - score_col: имя колонки с оценками (float).
    - min_cells: минимальное число клеток данного типа у донора.
    - threshold_method: 'mean_nsd', 'percentile', 'me_iqr', None.
    - n_sd: сколько SD прибавить к mean в методе 'mean_nsd'.
    - percentile: процентиль при методе 'percentile'.
    - draw_global: рисовать ли одну горизонтальную линию глобального порога.
    - global_threshold_kwargs: словарь того же метода, для глобального порога,
        например {'n_sd':2} или {'percentile':90}.
    """
    df = obs[obs[cluster_col].isin(cluster_val)].copy()

    # отфильтровать доноров по min_cells
    donor_counts = df["GSM_id"].value_counts()
    valid = donor_counts[donor_counts >= min_cells].index
    df = df[df["GSM_id"].isin(valid)]

    # подготовка меток доноров
    donor_info = (
        df[["GSM_id", "Age", "GSE_id"]]
        .drop_duplicates("GSM_id")
        .sort_values(by=["GSE_id", "Age"])
    )
    # donor_info['label'] = donor_info.apply(
    #     lambda r: f"{r['GSM_id']}\nAge:{int(r['Age'])}", axis=1)

    donor_info["label"] = [
        f"{gsm}\nAge:{int(age)}"
        for gsm, age in zip(donor_info["GSM_id"], donor_info["Age"])
    ]

    label_map = donor_info.set_index("GSM_id")["label"].to_dict()

    df["donor_label"] = pd.Categorical(
        df["GSM_id"].map(label_map), categories=list(donor_info["label"]), ordered=True
    )

    # функцию для порога
    def compute_threshold(arr, method, **kw):
        if len(arr) == 0:  # Проверка на пустой массив
            raise ValueError("Input array is empty, cannot compute threshold.")
        if method == "mean_nsd":
            return arr.mean() + kw.get("n_sd", n_sd) * arr.std()
        elif method == "percentile":
            return np.percentile(arr, kw.get("percentile", percentile))
        elif method == "me_iqr":
            # Проверка на минимальное количество данных для вычисления IQR
            if len(arr) < 2:
                raise ValueError(
                    "Not enough data to compute IQR (at least two data points are required)."
                )
            median = np.median(arr)
            q1 = np.percentile(arr, 25)  # 25-й процентиль
            q3 = np.percentile(arr, 75)  # 75-й процентиль
            iqr = q3 - q1
            return median + iqr
        else:
            raise ValueError("Unknown threshold_method")

    # пороги по донорам
    thresholds = {}
    for gsm, grp in df.groupby("GSM_id", observed=False):
        # Убедимся, что для данного донора есть данные в score_col
        if grp[score_col].dropna().empty:
            thresholds[gsm] = np.nan  # или любое другое значение по умолчанию
        else:
            threshold_kwargs = {
                k: v
                for k, v in {"n_sd": n_sd, "percentile": percentile}.items()
                if k in ["n_sd", "percentile"]
            }
            thresholds[gsm] = compute_threshold(
                grp[score_col], threshold_method, **threshold_kwargs
            )

    # глобальный порог
    global_thr = None
    if draw_global:
        global_thr = compute_threshold(
            df[score_col], threshold_method, **(global_threshold_kwargs or {})
        )

    df["GSE_id_str"] = df["GSE_id"].astype(str)
    df = df.sort_values(["donor_label", "GSE_id_str", "Age"])

    # рисуем
    plt.figure(figsize=(18, 6), dpi=150)
    ax = sns.violinplot(
        data=df,
        x="donor_label",
        y=score_col,
        hue="GSE_id",
        palette="tab10",
        fill=False,
        inner="quarters",
        linewidth=1,
        # size=2, jitter=True, alpha=0.4
    )
    sns.stripplot(
        data=df, x="donor_label", y=score_col, hue="GSE_id", ax=ax, size=2, jitter=True
    )

    # усреднённые статистики
    stats = df.groupby("donor_label", observed=False)[score_col].agg(
        ["mean", "median", "min", "max"]
    )
    # wiskers как [min, max]
    for i, label in enumerate(stats.index):
        m = stats.loc[label, "mean"]
        med = stats.loc[label, "median"]
        mn, mx = stats.loc[label, ["min", "max"]]
        # mean — зеленый треугольник
        ax.scatter(
            i, m, marker="D", color="green", s=50, label="mean" if i == 0 else ""
        )
        # median — синий крест
        ax.scatter(
            i, med, marker="X", color="blue", s=50, label="median" if i == 0 else ""
        )
        # whiskers
        ax.vlines(
            i, mn, mx, color="gray", linewidth=2, label="whiskers" if i == 0 else ""
        )

    # пороги
    for i, gsm in enumerate(donor_info["GSM_id"]):
        thr = thresholds[gsm]
        ax.hlines(
            y=thr,
            xmin=i - 0.4,
            xmax=i + 0.4,
            colors="red",
            linestyles="--",
            linewidth=2,
            label="donor thr" if i == 0 else "",
        )
    if global_thr is not None:
        ax.axhline(
            global_thr, color="magenta", linestyle="-", linewidth=2, label="global thr"
        )

    ax.set_xlabel("Donor\n(GSM_id и Age)", fontsize=12)
    ax.set_ylabel(score_col, fontsize=12)
    title = f"{cluster_val}: {score_col}\nThresholds: {threshold_method}"
    if draw_global:
        title += " (incl. global)"
    ax.set_title(title, fontsize=14)
    ax.tick_params(axis="x", labelrotation=90)
    ax.grid(True, axis="y", linestyle=":", linewidth=0.5)

    # легенда без дубликатов
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(
        by_label.values(),
        by_label.keys(),
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize="small",
    )

    plt.tight_layout()
    plt.show()


In [ ]:
obs_filt = adata.obs.dropna(subset=[score_colname])

In [ ]:
"""violin plots для score_all_cells с группировкой по GSE и bin=True"""

for t in obs_filt["Tissue_global"].unique().sort_values():
    _sub = obs_filt[obs_filt["Tissue_global"] == t]
    for ct in _sub["cell_type_final"].unique().sort_values():
        sub = _sub[_sub["cell_type_final"] == ct]
        print("\n", t, ct)
        plot_score_by_donor(
            obs=sub,
            cluster_col="cell_type_final",
            cluster_val=[ct],
            score_col=score_colname,
            min_cells=50,
            threshold_method="mean_nsd",
            n_sd=3,
            draw_global=False,
            # global_threshold_kwargs={'n_sd':2}
        )

In [ ]:
per_donor_list = []
for gsm in donors:
    sub = sub_ct[sub_ct["GSM_id"] == gsm]
    scores = sub[score_colname]
    age = int(sub["Age"].iloc[0])
    count = len(sub)
    if count <= 50:  # min cells
        continue
    per_donor_list.append((gsm, scores, age, count))

In [ ]:
"""Sen Score распределения для клеток подонороно - гистограммы"""
df = obs_filt.copy()

# 3) Цикл по кл типам
for t in df["Tissue_global"].unique().sort_values():
    _sub = df[df["Tissue_global"] == t]
    # просто по приколу с цветами далее поиграть
    rand_int = random.randint(0, 0xFFFFFF)
    hex_str = format(rand_int, "06x")

    for ct in _sub["cell_type_final"].unique().sort_values():
        sub_ct = _sub[_sub["cell_type_final"] == ct]
        donors = sub_ct.sort_values(by="Age")["GSM_id"].unique()

        n = len(donors)
        n_cols = min(5, n)
        n_rows = int(np.ceil(n / n_cols))

        fig, axes = plt.subplots(
            n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), constrained_layout=True
        )
        axes = axes.flatten()

        # общий диапазон по оси X для всех гистограмм
        x_min, x_max = sub_ct[score_colname].min(), sub_ct[score_colname].max()

        # 4) Построение гистограмм по донорам
        per_donor_list = []
        for gsm in donors:
            sub = sub_ct[sub_ct["GSM_id"] == gsm]
            scores = sub[score_colname]
            age = int(sub["Age"].iloc[0])
            count = len(sub)
            if count <= 50:  # min cells
                continue
            per_donor_list.append((gsm, scores, age, count))

        for ax, (gsm, scores, age, count) in zip(axes, per_donor_list):
            ax.hist(
                scores, bins=50, range=(x_min, x_max), color=f"#{hex_str}", alpha=0.7
            )
            ax.set_xlim(x_min, x_max)
            ax.set_title(f"{gsm} | Age {age}\nn={count}", fontsize=10)
            ax.set_xlabel(score_colname)
            ax.set_ylabel("Count")

        # отключить любые лишние оси
        for ax in axes[len(per_donor_list) :]:
            fig.delaxes(ax)

        # 5) Общий заголовок
        fig.suptitle(
            f"Distribution of {score_colname} for {t}-{ct} by donor", fontsize=14
        )
        plt.show()